# Method A with Restricted Label Space (Insect Orders Only)

Tests BioCLIP 2's zero-shot performance when the label space is restricted to species within the 14 target insect orders, rather than all species in BioCLIP's pretraining vocabulary.

**Purpose:** isolates whether restricting label space at the species level — keeping discriminative species strings but removing irrelevant orders — recovers performance, or whether the fundamental issue lies in BioCLIP 2's handling of short, prefix-sharing order-level strings.

**Comparison context:**
- **Method A standard** (Act 1): all species in BioCLIP's vocabulary → weighted F1 ≈ 0.861
- **Method A restricted** (this experiment): only species in 14 target orders → expected better
- **Method B** (Act 1): direct 14 short order strings → weighted F1 ≈ 0.014
- **Soft-prompts trained** (Act 2): learned discriminative tokens at prefix → weighted F1 ≈ 0.971

If Method A restricted substantially improves over Method A standard but doesn't reach soft-prompt levels, it confirms that label space matters but discriminative text representations remain the bottleneck for closed-set order classification.

**Outputs:**
- `output_method_a_restricted/per_class_results.csv` — per-class precision/recall/F1
- `output_method_a_restricted/summary.csv` — aggregate metrics

## Imports and configuration

In [1]:
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
import open_clip

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.8.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-80GB


In [2]:
CONFIG = {
    "model_name": "hf-hub:imageomics/bioclip-2",
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "meta_path": "/workspace/thesis/ultimate_bioclip_top1_rank_order_meta_12062025.csv",
    "image_root": "/workspace/thesis/ALL_copy",
    "output_dir": "/workspace/thesis/output_bioClip2_restricted_labels",

    "image_id_col": "image_id",
    "label_col": "label",

    "batch_size": 64,
    "num_workers": 4,

    # 14 target orders (present in your data)
    "target_orders": [
        "Araneae", "Blattodea", "Coleoptera", "Diptera", "Hemiptera",
        "Hymenoptera", "Ixodida", "Lepidoptera", "Mecoptera", "Neuroptera",
        "Orthoptera", "Plecoptera", "Raphidioptera", "Trichoptera",
    ],

    # Distractor orders: realistic taxonomic alternatives that BioCLIP could pick
    "distractor_orders": [
        # Other insect orders
        "Odonata", "Ephemeroptera", "Mantodea", "Phasmida",
        "Dermaptera", "Psocodea", "Thysanoptera", "Embioptera",
        "Strepsiptera", "Megaloptera", "Zygentoma", "Archaeognatha",
        "Siphonaptera",
        # Other Arachnida orders
        "Opiliones", "Scorpiones", "Pseudoscorpiones", "Solifugae",
        "Amblypygi", "Mesostigmata", "Trombidiformes", "Sarcoptiformes",
        # Other arthropods
        "Isopoda", "Amphipoda",
    ],

    # Full taxonomic templates (BioCLIP-style flattened hierarchy)
    "class_label_templates": {
        # 14 target orders
        "Blattodea":     "Animalia Arthropoda Insecta Blattodea",
        "Coleoptera":    "Animalia Arthropoda Insecta Coleoptera",
        "Diptera":       "Animalia Arthropoda Insecta Diptera",
        "Hemiptera":     "Animalia Arthropoda Insecta Hemiptera",
        "Hymenoptera":   "Animalia Arthropoda Insecta Hymenoptera",
        "Lepidoptera":   "Animalia Arthropoda Insecta Lepidoptera",
        "Mecoptera":     "Animalia Arthropoda Insecta Mecoptera",
        "Neuroptera":    "Animalia Arthropoda Insecta Neuroptera",
        "Orthoptera":    "Animalia Arthropoda Insecta Orthoptera",
        "Plecoptera":    "Animalia Arthropoda Insecta Plecoptera",
        "Raphidioptera": "Animalia Arthropoda Insecta Raphidioptera",
        "Trichoptera":   "Animalia Arthropoda Insecta Trichoptera",
        "Araneae":       "Animalia Arthropoda Arachnida Araneae",
        "Ixodida":       "Animalia Arthropoda Arachnida Ixodida",
        # Distractor orders — insects
        "Odonata":       "Animalia Arthropoda Insecta Odonata",
        "Ephemeroptera": "Animalia Arthropoda Insecta Ephemeroptera",
        "Mantodea":      "Animalia Arthropoda Insecta Mantodea",
        "Phasmida":      "Animalia Arthropoda Insecta Phasmida",
        "Dermaptera":    "Animalia Arthropoda Insecta Dermaptera",
        "Psocodea":      "Animalia Arthropoda Insecta Psocodea",
        "Thysanoptera":  "Animalia Arthropoda Insecta Thysanoptera",
        "Embioptera":    "Animalia Arthropoda Insecta Embioptera",
        "Strepsiptera":  "Animalia Arthropoda Insecta Strepsiptera",
        "Megaloptera":   "Animalia Arthropoda Insecta Megaloptera",
        "Zygentoma":     "Animalia Arthropoda Insecta Zygentoma",
        "Archaeognatha": "Animalia Arthropoda Insecta Archaeognatha",
        "Siphonaptera": "Animalia Arthropoda Insecta Siphonaptera",
        # Distractor orders — arachnids
        "Opiliones":        "Animalia Arthropoda Arachnida Opiliones",
        "Scorpiones":       "Animalia Arthropoda Arachnida Scorpiones",
        "Pseudoscorpiones": "Animalia Arthropoda Arachnida Pseudoscorpiones",
        "Solifugae":        "Animalia Arthropoda Arachnida Solifugae",
        "Amblypygi":        "Animalia Arthropoda Arachnida Amblypygi",
        "Mesostigmata":     "Animalia Arthropoda Arachnida Mesostigmata",
        "Trombidiformes":   "Animalia Arthropoda Arachnida Trombidiformes",
        "Sarcoptiformes":   "Animalia Arthropoda Arachnida Sarcoptiformes",
        # Distractor orders — other arthropods
        "Isopoda":   "Animalia Arthropoda Malacostraca Isopoda",
        "Amphipoda": "Animalia Arthropoda Malacostraca Amphipoda",
    },
}

# Build full label space as taxonomic strings (used by CustomLabelsClassifier)
CONFIG["full_label_space"] = [
    CONFIG["class_label_templates"][order]
    for order in (CONFIG["target_orders"] + CONFIG["distractor_orders"])
]

# Reverse mapping: from taxonomic string back to short order name
CONFIG["taxonomic_to_order"] = {
    v: k for k, v in CONFIG["class_label_templates"].items()
}

CROP_ID_PATTERN = re.compile(r"(\d{14}_crop_[a-zA-Z0-9]+\.jpg)$")
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)

print(f"Target orders ({len(CONFIG['target_orders'])}): {CONFIG['target_orders']}")
print(f"Distractor orders ({len(CONFIG['distractor_orders'])}): {CONFIG['distractor_orders']}")
print(f"Full label space size: {len(CONFIG['full_label_space'])}")
print(f"Example taxonomic string: {CONFIG['full_label_space'][0]}")
print(f"Output: {CONFIG['output_dir']}")

Target orders (14): ['Araneae', 'Blattodea', 'Coleoptera', 'Diptera', 'Hemiptera', 'Hymenoptera', 'Ixodida', 'Lepidoptera', 'Mecoptera', 'Neuroptera', 'Orthoptera', 'Plecoptera', 'Raphidioptera', 'Trichoptera']
Distractor orders (23): ['Odonata', 'Ephemeroptera', 'Mantodea', 'Phasmida', 'Dermaptera', 'Psocodea', 'Thysanoptera', 'Embioptera', 'Strepsiptera', 'Megaloptera', 'Zygentoma', 'Archaeognatha', 'Siphonaptera', 'Opiliones', 'Scorpiones', 'Pseudoscorpiones', 'Solifugae', 'Amblypygi', 'Mesostigmata', 'Trombidiformes', 'Sarcoptiformes', 'Isopoda', 'Amphipoda']
Full label space size: 37
Example taxonomic string: Animalia Arthropoda Arachnida Araneae
Output: /workspace/thesis/output_bioClip2_restricted_labels


## Building candidate species list

For "Method A restricted" ((BioClip2 with restricted label space), we need a list of species strings within the 14 target orders. We use BioCLIP 2's pretraining vocabulary (TreeOfLife taxonomic tree) and filter to keep only species whose order is in our target set.

The simplest source: pybioclip's bundled taxonomic data. If pybioclip is not installed, we can construct a representative species list from the model's known vocabulary by querying it programmatically.

For this experiment, we use the pybioclip library which has the taxonomic vocabulary built-in.

In [3]:
# Install pybioclip if not present
import subprocess
import sys

try:
    import bioclip
    print("bioclip already installed")
except ImportError:
    print("Installing pybioclip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pybioclip"])
    import bioclip
    print("Done.")

print(f"bioclip module loaded successfully")

bioclip already installed
bioclip module loaded successfully


In [4]:
from bioclip import CustomLabelsClassifier

# CustomLabelsClassifier accepts a specific list of class strings
classifier = CustomLabelsClassifier(
    cls_ary=CONFIG["full_label_space"],
    model_str=CONFIG["model_name"],
    device=CONFIG["device"],
)
print(f"Classifier loaded: {CONFIG['model_name']}")
print(f"Restricted to {len(CONFIG['full_label_space'])} classes")


Classifier loaded: hf-hub:imageomics/bioclip-2
Restricted to 37 classes


## Build path index and load metadata

In [5]:
def build_path_index(image_root):
    image_root = Path(image_root)
    files = list(image_root.glob("*.jpg"))
    path_index = {}
    for f in files:
        m = CROP_ID_PATTERN.search(f.name)
        if m:
            path_index[m.group(1)] = str(f)
    print(f"Mapped {len(path_index)} image_ids")
    return path_index


path_index = build_path_index(CONFIG["image_root"])

meta = pd.read_csv(CONFIG["meta_path"])
meta = meta.dropna(subset=[CONFIG["label_col"], CONFIG["image_id_col"]]).reset_index(drop=True)
meta["local_path"] = meta[CONFIG["image_id_col"]].map(path_index)
matched = meta.dropna(subset=["local_path"]).reset_index(drop=True)

y_str = matched[CONFIG["label_col"]].to_numpy()
image_paths = matched["local_path"].to_numpy()
classnames = sorted(np.unique(y_str).tolist())

print(f"Matched: {len(matched)} crops")
print(f"Classes ({len(classnames)}): {classnames}")

Mapped 48095 image_ids
Matched: 39787 crops
Classes (14): ['Araneae', 'Blattodea', 'Coleoptera', 'Diptera', 'Hemiptera', 'Hymenoptera', 'Ixodida', 'Lepidoptera', 'Mecoptera', 'Neuroptera', 'Orthoptera', 'Plecoptera', 'Raphidioptera', 'Trichoptera']


## Running Method A restricted: classify with order rank, restricted to target orders

pybioclip's `TreeOfLifeClassifier.predict()` with `rank=Rank.ORDER` aggregates over species within each order. By setting `class_str` to our 14 target orders, the classifier compares image features against the species under those orders only (not all species in BioCLIP's vocabulary).

This is the **restricted Method A**: same species-level comparison as standard Method A, but the label space is filtered to species within our 14 target orders.

In [6]:
print(f"Running Method A restricted on {len(matched)} images...")
print(f"Label space: {len(CONFIG['target_orders'])} target orders + "
      f"{len(CONFIG['distractor_orders'])} distractor orders = "
      f"{len(CONFIG['full_label_space'])} total orders\n")

start = time.time()
predictions = []
predicted_distractor_count = 0

batch_size = CONFIG["batch_size"]
n_batches = (len(image_paths) + batch_size - 1) // batch_size

for batch_idx in range(n_batches):
    batch_start = batch_idx * batch_size
    batch_end = min(batch_start + batch_size, len(image_paths))
    batch_paths = image_paths[batch_start:batch_end].tolist()

    # CustomLabelsClassifier — class list already provided at init, only k= parameter here
    results = classifier.predict(
        images=batch_paths,
        k=1,
    )

    for image_path in batch_paths:
        image_results = [r for r in results if r["file_name"] == image_path]
        if image_results:
            # CustomLabelsClassifier returns the classification field, not order
            predicted_full_string = image_results[0]["classification"]
            # Map from taxonomic string back to short order name
            predicted_order = CONFIG["taxonomic_to_order"].get(predicted_full_string, None)
            if predicted_order in CONFIG["distractor_orders"]:
                predicted_distractor_count += 1
        else:
            predicted_order = None
        predictions.append(predicted_order)

    if batch_idx % 50 == 0:
        elapsed = time.time() - start
        eta = elapsed / (batch_idx + 1) * (n_batches - batch_idx - 1)
        print(f"  Batch {batch_idx+1}/{n_batches}: "
              f"elapsed={timedelta(seconds=int(elapsed))}, "
              f"ETA={timedelta(seconds=int(eta))}")

total_time = time.time() - start
print(f"\nTotal inference time: {timedelta(seconds=int(total_time))}")
print(f"Predictions made: {len(predictions)}")
print(f"Predicted as distractor: {predicted_distractor_count} "
      f"({100*predicted_distractor_count/len(predictions):.2f}%)")

Running Method A restricted on 39787 images...
Label space: 14 target orders + 23 distractor orders = 37 total orders



100%|██████████| 64/64 [00:38<00:00,  1.67images/s]


  Batch 1/622: elapsed=0:00:38, ETA=6:37:36


100%|██████████| 64/64 [00:37<00:00,  1.68images/s]


  Batch 51/622: elapsed=0:34:17, ETA=6:23:59


...

  Batch 551/622: elapsed=6:08:18, ETA=0:47:27


100%|██████████| 64/64 [00:38<00:00,  1.66images/s]


  Batch 601/622: elapsed=6:41:48, ETA=0:14:02


100%|██████████| 43/43 [00:28<00:00,  1.53images/s]


Total inference time: 6:55:21
Predictions made: 39787
Predicted as distractor: 6540 (16.44%)


## Evaluate predictions

In [7]:
# Converting predictions to numpy and compute report
y_true = y_str
y_pred = np.array(predictions)

# Handling any None predictions (shouldn't happen but just in case)
valid_mask = y_pred != None
print(f"Valid predictions: {valid_mask.sum()}/{len(y_pred)}")
y_true_valid = y_true[valid_mask]
y_pred_valid = y_pred[valid_mask]

# Classification report
report = classification_report(
    y_true_valid, y_pred_valid,
    labels=classnames, target_names=classnames,
    output_dict=True, zero_division=0,
)

# Per-class results
per_class_rows = []
for cls_name in classnames:
    stats = report.get(cls_name, {})
    per_class_rows.append({
        "class": cls_name,
        "precision": round(stats.get("precision", 0.0), 4),
        "recall": round(stats.get("recall", 0.0), 4),
        "f1": round(stats.get("f1-score", 0.0), 4),
        "support": int(stats.get("support", 0)),
    })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(Path(CONFIG["output_dir"]) / "per_class_results.csv", index=False)
print("\n=== Per-class results ===")
print(per_class_df.to_string(index=False))

Valid predictions: 39787/39787

=== Per-class results ===
        class  precision  recall     f1  support
      Araneae     0.9570  0.9468 0.9519      564
    Blattodea     0.1570  0.8261 0.2639       23
   Coleoptera     0.2180  0.2625 0.2382     1143
      Diptera     0.9931  0.7689 0.8667    31913
    Hemiptera     0.1875  0.0774 0.1096      775
  Hymenoptera     0.7744  0.8019 0.7879     3326
      Ixodida     0.0041  1.0000 0.0082        1
  Lepidoptera     0.9530  0.6858 0.7976     1744
    Mecoptera     0.0968  0.0882 0.0923       34
   Neuroptera     0.3333  0.0261 0.0484      115
   Orthoptera     0.0576  1.0000 0.1088        8
   Plecoptera     0.1049  0.9074 0.1881       54
Raphidioptera     0.2222  1.0000 0.3636        2
  Trichoptera     0.1235  0.8235 0.2147       85


In [8]:
# Aggregate metrics
weighted_f1 = report["weighted avg"]["f1-score"]
macro_f1_all = report["macro avg"]["f1-score"]

# macro F1 restricted to n>=50 classes
eligible_classes = [c for c in classnames if (y_str == c).sum() >= 50]
eligible_f1s = [report[c]["f1-score"] for c in eligible_classes if c in report]
macro_f1_restricted = float(np.mean(eligible_f1s)) if eligible_f1s else 0.0

hemiptera_f1 = report.get("Hemiptera", {}).get("f1-score", 0.0)
coleoptera_f1 = report.get("Coleoptera", {}).get("f1-score", 0.0)

summary_df = pd.DataFrame([{
    "method": "method_a_restricted",
    "weighted_f1": round(weighted_f1, 4),
    "macro_f1_all": round(macro_f1_all, 4),
    "macro_f1_restricted_n50": round(macro_f1_restricted, 4),
    "Hemiptera_f1": round(hemiptera_f1, 4),
    "Coleoptera_f1": round(coleoptera_f1, 4),
    "inference_time_sec": int(total_time),
}])
summary_df.to_csv(Path(CONFIG["output_dir"]) / "summary.csv", index=False)
print("\n=== Summary ===")
print(summary_df.to_string(index=False))


=== Summary ===
             method  weighted_f1  macro_f1_all  macro_f1_restricted_n50  Hemiptera_f1  Coleoptera_f1  inference_time_sec
method_a_restricted       0.8196          0.36                    0.467        0.1096         0.2382               24921


## Output files summary

| File | Purpose |
|---|---|
| `per_class_results.csv` | Per-class precision/recall/F1 |
| `summary.csv` | Aggregate metrics (weighted F1, macro F1, Hemiptera F1, Coleoptera F1) |
